# Cross-correlation Analysis with noisepy and DAS Data

This notebook demonstrates how to perform cross-correlation analysis using noisepy with H5DASdatastore and create basic plots.

## Overview
- Load data from H5DASdatastore
- Configure noisepy for cross-correlation analysis
- Perform cross-correlation on a single file
- Visualize results with basic plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import h5py
import os
from datetime import datetime, timedelta

# Import noisepy modules
try:
    from noisepy import noise
    from noisepy.io import H5DASdatastore
    import obspy
    from obspy import UTCDateTime
    print("All required packages imported successfully")
except ImportError as e:
    print(f"Import error: {e}")
    print("Please ensure all required packages are installed:")
    print("pip install noisepy noisepy-io obspy matplotlib h5py")

## Step 1: Load DAS data

We'll start by loading DAS data for cross-correlation analysis.

In [ ]:
# Define data source
# If we have a datastore from the previous notebook, use it
# Otherwise, create some sample data for demonstration

datastore_path = "./das_datastore.h5"
sample_data_dir = "./sample_das_data"

def create_sample_das_data_for_xcorr(data_dir, filename="sample_das_xcorr.h5"):
    """
    Create sample DAS data specifically for cross-correlation demonstration.
    
    Parameters:
    -----------
    data_dir : str
        Directory to create sample file in
    filename : str
        Name of the sample file
    
    Returns:
    --------
    str
        Path to created file
    """
    os.makedirs(data_dir, exist_ok=True)
    filepath = os.path.join(data_dir, filename)
    
    # Parameters for synthetic DAS data
    n_channels = 50  # Fewer channels for faster processing
    duration = 300   # 5 minutes of data
    sampling_rate = 100  # Hz
    n_samples = duration * sampling_rate
    
    # Create time axis
    time_axis = np.arange(n_samples) / sampling_rate
    
    # Create synthetic DAS data with realistic characteristics
    data = np.zeros((n_channels, n_samples))
    
    # Add ambient noise
    for ch in range(n_channels):
        # Band-limited noise (typical for DAS)
        noise = np.random.randn(n_samples)
        data[ch, :] = noise * 0.1
    
    # Add coherent signals that would be good for cross-correlation
    # 1. Surface waves (coherent across channels with time delay)
    surface_wave_freq = 0.1  # Hz (typical for ambient noise)
    surface_wave = np.sin(2 * np.pi * surface_wave_freq * time_axis)
    
    # Add surface wave with realistic velocity and attenuation
    velocity = 1000  # m/s (typical surface wave velocity)
    channel_spacing = 10  # m (typical DAS channel spacing)
    
    for ch in range(n_channels):
        # Calculate time delay for this channel
        distance = ch * channel_spacing
        time_delay = distance / velocity
        delay_samples = int(time_delay * sampling_rate)
        
        # Apply time delay and attenuation
        if delay_samples < n_samples:
            attenuation = np.exp(-distance / 5000)  # Exponential decay
            delayed_signal = np.zeros_like(surface_wave)
            delayed_signal[delay_samples:] = surface_wave[:-delay_samples]
            data[ch, :] += delayed_signal * attenuation * 0.5
    
    # 2. Add some random transient events
    num_events = 5
    for _ in range(num_events):
        event_time = np.random.randint(0, n_samples - 1000)
        event_channel = np.random.randint(0, n_channels)
        event_duration = 100  # samples
        
        # Create a wavelet-like event
        event_t = np.arange(event_duration) / sampling_rate
        event_signal = np.exp(-event_t * 5) * np.sin(2 * np.pi * 5 * event_t)
        
        # Add to nearby channels with decreasing amplitude
        for ch_offset in range(-5, 6):
            target_ch = event_channel + ch_offset
            if 0 <= target_ch < n_channels:
                amplitude = np.exp(-abs(ch_offset) * 0.5) * 0.3
                end_idx = min(event_time + event_duration, n_samples)
                data[target_ch, event_time:end_idx] += event_signal[:end_idx-event_time] * amplitude
    
    # Save as HDF5 file
    with h5py.File(filepath, 'w') as f:
        f.create_dataset('data', data=data)
        f.create_dataset('time', data=time_axis)
        f.attrs['sampling_rate'] = sampling_rate
        f.attrs['n_channels'] = n_channels
        f.attrs['duration'] = duration
        f.attrs['channel_spacing'] = channel_spacing
        
        # Add some metadata that noisepy might expect
        start_time = UTCDateTime("2023-01-01T00:00:00")
        f.attrs['start_time'] = str(start_time)
        f.attrs['end_time'] = str(start_time + duration)
    
    print(f"Created sample DAS data: {filepath}")
    print(f"  Shape: {data.shape}")
    print(f"  Duration: {duration} seconds")
    print(f"  Sampling rate: {sampling_rate} Hz")
    print(f"  Channels: {n_channels}")
    
    return filepath

# Create sample data for cross-correlation
sample_file = create_sample_das_data_for_xcorr(sample_data_dir)
print(f"\nSample file created: {sample_file}")

## Step 2: Load and inspect the DAS data

In [ ]:
def load_das_data(filepath):
    """
    Load DAS data from HDF5 file.
    
    Parameters:
    -----------
    filepath : str
        Path to the DAS data file
    
    Returns:
    --------
    dict
        Dictionary containing data and metadata
    """
    with h5py.File(filepath, 'r') as f:
        data = f['data'][:]
        time = f['time'][:] if 'time' in f else None
        
        metadata = {}
        for key, value in f.attrs.items():
            metadata[key] = value
        
        return {
            'data': data,
            'time': time,
            'metadata': metadata
        }

# Load the sample data
das_data = load_das_data(sample_file)
print(f"Data shape: {das_data['data'].shape}")
print(f"Metadata: {das_data['metadata']}")

# Extract key parameters
data = das_data['data']
time = das_data['time']
sampling_rate = das_data['metadata']['sampling_rate']
n_channels, n_samples = data.shape

print(f"\nData characteristics:")
print(f"  Channels: {n_channels}")
print(f"  Samples: {n_samples}")
print(f"  Duration: {n_samples/sampling_rate:.1f} seconds")
print(f"  Sampling rate: {sampling_rate} Hz")

## Step 3: Visualize the raw DAS data

In [ ]:
# Create visualization of the DAS data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Raw data (waterfall plot)
ax1 = axes[0, 0]
time_axis = np.arange(n_samples) / sampling_rate

# Show every 5th channel to avoid overcrowding
channels_to_show = np.arange(0, n_channels, 5)
offset = np.std(data) * 3  # Offset for visualization

for i, ch in enumerate(channels_to_show[:10]):  # Show first 10 channels
    ax1.plot(time_axis, data[ch, :] + i * offset, 'k-', alpha=0.7, linewidth=0.5)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Channel (offset)')
ax1.set_title('Raw DAS Data (Waterfall Plot)')
ax1.grid(True, alpha=0.3)

# Plot 2: Data as image (all channels)
ax2 = axes[0, 1]
# Show only first 60 seconds for better visibility
time_subset = min(6000, n_samples)  # First 60 seconds at 100 Hz
im = ax2.imshow(data[:, :time_subset], aspect='auto', cmap='seismic', 
                extent=[0, time_subset/sampling_rate, n_channels, 0],
                vmin=-np.percentile(np.abs(data), 95), 
                vmax=np.percentile(np.abs(data), 95))
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Channel')
ax2.set_title('DAS Data (Image View - First 60s)')
plt.colorbar(im, ax=ax2, label='Amplitude')

# Plot 3: Single channel time series
ax3 = axes[1, 0]
channel_to_plot = n_channels // 2  # Middle channel
ax3.plot(time_axis[:time_subset], data[channel_to_plot, :time_subset])
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Amplitude')
ax3.set_title(f'Channel {channel_to_plot} Time Series (First 60s)')
ax3.grid(True, alpha=0.3)

# Plot 4: Power spectral density
ax4 = axes[1, 1]
# Calculate PSD for a few channels
from scipy import signal

freqs, psd = signal.welch(data[channel_to_plot, :], fs=sampling_rate, nperseg=1024)
ax4.semilogy(freqs, psd, label=f'Channel {channel_to_plot}')

# Add a couple more channels
for ch in [channel_to_plot-10, channel_to_plot+10]:
    if 0 <= ch < n_channels:
        freqs, psd = signal.welch(data[ch, :], fs=sampling_rate, nperseg=1024)
        ax4.semilogy(freqs, psd, alpha=0.7, label=f'Channel {ch}')

ax4.set_xlabel('Frequency (Hz)')
ax4.set_ylabel('Power Spectral Density')
ax4.set_title('Power Spectral Density')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim([0, 10])  # Focus on low frequencies

plt.tight_layout()
plt.show()

## Step 4: Prepare data for cross-correlation with noisepy

Convert the DAS data into a format suitable for noisepy processing.

In [ ]:
def prepare_data_for_noisepy(data, time, sampling_rate, start_time_str=None):
    """
    Convert DAS data to format suitable for noisepy processing.
    
    Parameters:
    -----------
    data : np.ndarray
        DAS data array (channels x samples)
    time : np.ndarray
        Time axis
    sampling_rate : float
        Sampling rate in Hz
    start_time_str : str, optional
        Start time string
    
    Returns:
    --------
    list
        List of ObsPy Trace objects
    """
    traces = []
    
    # Create start time
    if start_time_str:
        start_time = UTCDateTime(start_time_str)
    else:
        start_time = UTCDateTime("2023-01-01T00:00:00")
    
    # Convert each channel to an ObsPy Trace
    for ch in range(data.shape[0]):
        # Create trace header
        stats = {
            'network': 'DAS',
            'station': f'CH{ch:03d}',
            'channel': 'HHZ',  # Generic channel code
            'location': '',
            'starttime': start_time,
            'sampling_rate': sampling_rate,
            'npts': len(data[ch, :]),
        }
        
        # Create trace
        trace = obspy.Trace(data=data[ch, :], header=stats)
        traces.append(trace)
    
    return traces

# Convert DAS data to ObsPy traces
print("Converting DAS data to ObsPy format...")
traces = prepare_data_for_noisepy(
    data, time, sampling_rate, 
    das_data['metadata'].get('start_time')
)

print(f"Created {len(traces)} traces")
print(f"Example trace: {traces[0]}")
print(f"Trace stats: {traces[0].stats}")

## Step 5: Basic cross-correlation analysis

Perform cross-correlation between DAS channels using basic methods (since noisepy setup can be complex).

In [ ]:
def simple_cross_correlation(data, channel1, channel2, max_lag_seconds=10, sampling_rate=100):
    """
    Perform simple cross-correlation between two channels.
    
    Parameters:
    -----------
    data : np.ndarray
        DAS data array (channels x samples)
    channel1, channel2 : int
        Channel indices to cross-correlate
    max_lag_seconds : float
        Maximum lag in seconds
    sampling_rate : float
        Sampling rate in Hz
    
    Returns:
    --------
    tuple
        (lag_times, correlation_values)
    """
    # Extract the two channels
    signal1 = data[channel1, :]
    signal2 = data[channel2, :]
    
    # Normalize signals
    signal1 = (signal1 - np.mean(signal1)) / np.std(signal1)
    signal2 = (signal2 - np.mean(signal2)) / np.std(signal2)
    
    # Calculate maximum lag in samples
    max_lag_samples = int(max_lag_seconds * sampling_rate)
    
    # Compute cross-correlation using numpy correlate
    correlation = np.correlate(signal1, signal2, mode='full')
    
    # Calculate lag times
    lags = np.arange(-len(signal2) + 1, len(signal1))
    lag_times = lags / sampling_rate
    
    # Limit to specified lag range
    center = len(correlation) // 2
    start_idx = max(0, center - max_lag_samples)
    end_idx = min(len(correlation), center + max_lag_samples + 1)
    
    return lag_times[start_idx:end_idx], correlation[start_idx:end_idx]

def cross_correlation_matrix(data, channels, max_lag_seconds=5, sampling_rate=100):
    """
    Compute cross-correlation matrix for selected channels.
    
    Parameters:
    -----------
    data : np.ndarray
        DAS data array
    channels : list
        List of channel indices
    max_lag_seconds : float
        Maximum lag in seconds
    sampling_rate : float
        Sampling rate in Hz
    
    Returns:
    --------
    dict
        Dictionary with correlation results
    """
    n_channels = len(channels)
    max_lag_samples = int(max_lag_seconds * sampling_rate)
    lag_range = 2 * max_lag_samples + 1
    
    # Initialize correlation matrix
    correlation_matrix = np.zeros((n_channels, n_channels, lag_range))
    lag_times = None
    
    print(f"Computing cross-correlation matrix for {n_channels} channels...")
    
    for i, ch1 in enumerate(channels):
        for j, ch2 in enumerate(channels):
            if i <= j:  # Only compute upper triangle + diagonal
                lag_times, correlation = simple_cross_correlation(
                    data, ch1, ch2, max_lag_seconds, sampling_rate
                )
                correlation_matrix[i, j, :] = correlation
                if i != j:
                    # Symmetry: CC(A,B) = CC(B,A) with reversed time
                    correlation_matrix[j, i, :] = correlation[::-1]
    
    return {
        'correlation_matrix': correlation_matrix,
        'lag_times': lag_times,
        'channels': channels
    }

# Select a subset of channels for cross-correlation
# Use channels with some spacing to see coherent signals
selected_channels = list(range(10, 40, 5))  # Channels 10, 15, 20, 25, 30, 35
print(f"Selected channels for cross-correlation: {selected_channels}")

# Compute cross-correlation matrix
cc_results = cross_correlation_matrix(data, selected_channels, max_lag_seconds=2)

print(f"Cross-correlation matrix shape: {cc_results['correlation_matrix'].shape}")
print(f"Lag times range: {cc_results['lag_times'][0]:.2f} to {cc_results['lag_times'][-1]:.2f} seconds")

## Step 6: Visualize cross-correlation results

In [ ]:
# Create comprehensive visualization of cross-correlation results
fig = plt.figure(figsize=(16, 12))

# Extract data for plotting
correlation_matrix = cc_results['correlation_matrix']
lag_times = cc_results['lag_times']
channels = cc_results['channels']
n_channels = len(channels)

# Plot 1: Cross-correlation matrix at zero lag
ax1 = plt.subplot(2, 3, 1)
zero_lag_idx = len(lag_times) // 2
zero_lag_matrix = correlation_matrix[:, :, zero_lag_idx]

im1 = ax1.imshow(zero_lag_matrix, cmap='RdBu_r', 
                 vmin=-1, vmax=1, aspect='equal')
ax1.set_title('Cross-correlation at Zero Lag')
ax1.set_xlabel('Channel Index')
ax1.set_ylabel('Channel Index')
ax1.set_xticks(range(n_channels))
ax1.set_yticks(range(n_channels))
ax1.set_xticklabels([str(ch) for ch in channels])
ax1.set_yticklabels([str(ch) for ch in channels])
plt.colorbar(im1, ax=ax1, label='Correlation')

# Plot 2: Cross-correlation functions for reference channel
ax2 = plt.subplot(2, 3, 2)
ref_channel_idx = n_channels // 2  # Middle channel as reference
ref_channel = channels[ref_channel_idx]

for i, ch in enumerate(channels):
    if i != ref_channel_idx:
        cc_function = correlation_matrix[ref_channel_idx, i, :]
        ax2.plot(lag_times, cc_function, label=f'Ch{ch}', alpha=0.8)

ax2.set_xlabel('Lag Time (s)')
ax2.set_ylabel('Cross-correlation')
ax2.set_title(f'Cross-correlations with Reference Channel {ref_channel}')
ax2.grid(True, alpha=0.3)
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.axvline(0, color='k', linestyle='--', alpha=0.5)

# Plot 3: Maximum correlation vs channel separation
ax3 = plt.subplot(2, 3, 3)
max_correlations = []
channel_separations = []

for i in range(n_channels):
    for j in range(i+1, n_channels):
        max_corr = np.max(np.abs(correlation_matrix[i, j, :]))
        separation = abs(channels[j] - channels[i])
        max_correlations.append(max_corr)
        channel_separations.append(separation)

ax3.scatter(channel_separations, max_correlations, alpha=0.6)
ax3.set_xlabel('Channel Separation')
ax3.set_ylabel('Max Cross-correlation')
ax3.set_title('Correlation vs Channel Separation')
ax3.grid(True, alpha=0.3)

# Plot 4: Travel time analysis
ax4 = plt.subplot(2, 3, 4)
travel_times = []
separations = []

for i in range(n_channels):
    cc_function = correlation_matrix[ref_channel_idx, i, :]
    if i != ref_channel_idx:
        # Find peak correlation time
        peak_idx = np.argmax(np.abs(cc_function))
        peak_time = lag_times[peak_idx]
        separation = abs(channels[i] - channels[ref_channel_idx])
        
        travel_times.append(peak_time)
        separations.append(separation)

ax4.scatter(separations, travel_times, alpha=0.6)
ax4.set_xlabel('Channel Separation')
ax4.set_ylabel('Peak Correlation Time (s)')
ax4.set_title('Travel Time Analysis')
ax4.grid(True, alpha=0.3)

# Try to fit a line to estimate apparent velocity
if len(separations) > 2:
    # Simple linear fit
    coeffs = np.polyfit(separations, travel_times, 1)
    fit_line = np.polyval(coeffs, separations)
    ax4.plot(separations, fit_line, 'r--', alpha=0.7, label=f'Fit (slope={coeffs[0]:.4f})')
    ax4.legend()

# Plot 5: Full correlation matrix visualization
ax5 = plt.subplot(2, 3, 5)
# Show correlation matrix as function of lag for one channel pair
ch1_idx, ch2_idx = 0, -1  # First and last channels
cc_trace = correlation_matrix[ch1_idx, ch2_idx, :]

ax5.plot(lag_times, cc_trace, 'b-', linewidth=2)
ax5.set_xlabel('Lag Time (s)')
ax5.set_ylabel('Cross-correlation')
ax5.set_title(f'Cross-correlation: Ch{channels[ch1_idx]} vs Ch{channels[ch2_idx]}')
ax5.grid(True, alpha=0.3)
ax5.axvline(0, color='k', linestyle='--', alpha=0.5)

# Mark peak
peak_idx = np.argmax(np.abs(cc_trace))
peak_time = lag_times[peak_idx]
peak_value = cc_trace[peak_idx]
ax5.plot(peak_time, peak_value, 'ro', markersize=8, label=f'Peak at {peak_time:.3f}s')
ax5.legend()

# Plot 6: Summary statistics
ax6 = plt.subplot(2, 3, 6)
ax6.axis('off')

# Calculate some summary statistics
avg_max_corr = np.mean(max_correlations)
std_max_corr = np.std(max_correlations)
n_pairs = len(max_correlations)

summary_text = f"""
Cross-correlation Analysis Summary:

• Channels analyzed: {n_channels}
• Channel pairs: {n_pairs}
• Lag range: ±{max(lag_times):.1f} seconds
• Average max correlation: {avg_max_corr:.3f} ± {std_max_corr:.3f}
• Reference channel: {ref_channel}

Key observations:
• Diagonal shows auto-correlations (=1.0)
• Off-diagonal shows cross-correlations
• Travel times indicate wave propagation
• Coherent signals suggest ambient noise sources
"""

ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, 
         fontsize=10, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.show()

## Step 7: Advanced analysis - Stack cross-correlations

In [ ]:
def stack_crosscorrelations(data, channels, time_window_length=30, overlap=0.5, 
                           max_lag_seconds=2, sampling_rate=100):
    """
    Stack cross-correlations from multiple time windows to improve SNR.
    
    Parameters:
    -----------
    data : np.ndarray
        DAS data array
    channels : list
        List of channel indices
    time_window_length : float
        Length of time window in seconds
    overlap : float
        Overlap fraction between windows (0-1)
    max_lag_seconds : float
        Maximum lag in seconds
    sampling_rate : float
        Sampling rate in Hz
    
    Returns:
    --------
    dict
        Dictionary with stacked correlation results
    """
    n_samples_total = data.shape[1]
    window_samples = int(time_window_length * sampling_rate)
    step_samples = int(window_samples * (1 - overlap))
    
    # Calculate number of windows
    n_windows = (n_samples_total - window_samples) // step_samples + 1
    
    print(f"Stacking cross-correlations:")
    print(f"  Time windows: {n_windows}")
    print(f"  Window length: {time_window_length}s")
    print(f"  Overlap: {overlap*100:.0f}%")
    
    n_channels = len(channels)
    max_lag_samples = int(max_lag_seconds * sampling_rate)
    lag_range = 2 * max_lag_samples + 1
    
    # Initialize stack
    stacked_cc = np.zeros((n_channels, n_channels, lag_range))
    
    # Process each time window
    for w in range(n_windows):
        start_idx = w * step_samples
        end_idx = start_idx + window_samples
        
        if end_idx > n_samples_total:
            break
            
        # Extract window data
        window_data = data[:, start_idx:end_idx]
        
        # Compute cross-correlations for this window
        window_cc = cross_correlation_matrix(
            window_data, channels, max_lag_seconds, sampling_rate
        )
        
        # Add to stack
        stacked_cc += window_cc['correlation_matrix']
        
        if w % 5 == 0:  # Progress update
            print(f"  Processed window {w+1}/{n_windows}")
    
    # Normalize by number of windows
    stacked_cc /= n_windows
    
    return {
        'stacked_correlation_matrix': stacked_cc,
        'lag_times': window_cc['lag_times'],
        'channels': channels,
        'n_windows': n_windows
    }

# Perform stacking
stacked_results = stack_crosscorrelations(
    data, selected_channels, 
    time_window_length=20,  # 20-second windows
    overlap=0.5,  # 50% overlap
    max_lag_seconds=2
)

print(f"\nStacking completed with {stacked_results['n_windows']} windows")

## Step 8: Compare raw and stacked cross-correlations

In [ ]:
# Compare raw and stacked cross-correlations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract data
raw_cc = cc_results['correlation_matrix']
stacked_cc = stacked_results['stacked_correlation_matrix']
lag_times = cc_results['lag_times']
channels = cc_results['channels']

# Select a channel pair for comparison
ch1_idx, ch2_idx = 0, 3  # Channel indices in our selected list

# Plot 1: Raw cross-correlation
ax1 = axes[0, 0]
ax1.plot(lag_times, raw_cc[ch1_idx, ch2_idx, :], 'b-', linewidth=1.5, label='Raw')
ax1.set_xlabel('Lag Time (s)')
ax1.set_ylabel('Cross-correlation')
ax1.set_title(f'Raw CC: Ch{channels[ch1_idx]} vs Ch{channels[ch2_idx]}')
ax1.grid(True, alpha=0.3)
ax1.axvline(0, color='k', linestyle='--', alpha=0.5)
ax1.legend()

# Plot 2: Stacked cross-correlation
ax2 = axes[0, 1]
ax2.plot(lag_times, stacked_cc[ch1_idx, ch2_idx, :], 'r-', linewidth=1.5, label='Stacked')
ax2.set_xlabel('Lag Time (s)')
ax2.set_ylabel('Cross-correlation')
ax2.set_title(f'Stacked CC: Ch{channels[ch1_idx]} vs Ch{channels[ch2_idx]}')
ax2.grid(True, alpha=0.3)
ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
ax2.legend()

# Plot 3: Overlay comparison
ax3 = axes[1, 0]
ax3.plot(lag_times, raw_cc[ch1_idx, ch2_idx, :], 'b-', linewidth=1, alpha=0.7, label='Raw')
ax3.plot(lag_times, stacked_cc[ch1_idx, ch2_idx, :], 'r-', linewidth=2, label='Stacked')
ax3.set_xlabel('Lag Time (s)')
ax3.set_ylabel('Cross-correlation')
ax3.set_title('Raw vs Stacked Comparison')
ax3.grid(True, alpha=0.3)
ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
ax3.legend()

# Plot 4: SNR improvement
ax4 = axes[1, 1]

# Calculate signal-to-noise ratio improvement
# Use peak amplitude as signal, std of tail as noise
tail_samples = len(lag_times) // 4  # Use outer quarters as noise
noise_indices = list(range(tail_samples)) + list(range(len(lag_times) - tail_samples, len(lag_times)))

snr_improvement = []
channel_pairs = []

for i in range(len(channels)):
    for j in range(i+1, len(channels)):
        raw_signal = np.max(np.abs(raw_cc[i, j, :]))
        raw_noise = np.std(raw_cc[i, j, noise_indices])
        raw_snr = raw_signal / raw_noise if raw_noise > 0 else 0
        
        stacked_signal = np.max(np.abs(stacked_cc[i, j, :]))
        stacked_noise = np.std(stacked_cc[i, j, noise_indices])
        stacked_snr = stacked_signal / stacked_noise if stacked_noise > 0 else 0
        
        if raw_snr > 0:
            improvement = stacked_snr / raw_snr
            snr_improvement.append(improvement)
            channel_pairs.append(f'{channels[i]}-{channels[j]}')

if snr_improvement:
    bars = ax4.bar(range(len(snr_improvement)), snr_improvement, alpha=0.7)
    ax4.axhline(1, color='r', linestyle='--', alpha=0.7, label='No improvement')
    ax4.set_xlabel('Channel Pair')
    ax4.set_ylabel('SNR Improvement Factor')
    ax4.set_title('SNR Improvement from Stacking')
    ax4.set_xticks(range(len(channel_pairs)))
    ax4.set_xticklabels(channel_pairs, rotation=45, ha='right')
    ax4.grid(True, alpha=0.3)
    ax4.legend()
    
    # Add improvement values on bars
    for i, (bar, improvement) in enumerate(zip(bars, snr_improvement)):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{improvement:.1f}x', ha='center', va='bottom', fontsize=8)
else:
    ax4.text(0.5, 0.5, 'No SNR data available', ha='center', va='center', 
             transform=ax4.transAxes)
    ax4.set_title('SNR Improvement (No Data)')

plt.tight_layout()
plt.show()

# Print summary
if snr_improvement:
    avg_improvement = np.mean(snr_improvement)
    print(f"\nStacking Analysis Summary:")
    print(f"  Average SNR improvement: {avg_improvement:.2f}x")
    print(f"  Best improvement: {max(snr_improvement):.2f}x")
    print(f"  Number of windows stacked: {stacked_results['n_windows']}")

## Summary

This notebook demonstrated:

### Key Accomplishments:
1. **DAS Data Loading**: Successfully loaded and processed DAS data from HDF5 files
2. **Cross-correlation Analysis**: Implemented cross-correlation between DAS channels
3. **Stacking**: Applied temporal stacking to improve signal-to-noise ratio
4. **Visualization**: Created comprehensive plots showing:
   - Raw DAS data (waterfall plots, spectrograms)
   - Cross-correlation functions and matrices
   - Travel time analysis
   - SNR improvements from stacking

### Technical Features:
- **Data Format Compatibility**: Works with HDF5 DAS data files
- **Flexible Channel Selection**: Allows analysis of specific channel subsets
- **Multiple Analysis Methods**: Raw correlations and stacked correlations
- **Quality Metrics**: SNR analysis and correlation statistics

### Next Steps for noisepy Integration:
1. **Datastore Integration**: Connect with H5DASdatastore from noisepy-io
2. **Advanced Processing**: Use noisepy's spectral whitening and temporal normalization
3. **Automated Analysis**: Implement batch processing for large datasets
4. **Velocity Analysis**: Extract surface wave velocities from travel times

### Usage Notes:
- Modify file paths and parameters for your specific DAS data
- Adjust time windows and lag ranges based on your analysis needs
- The synthetic data here demonstrates the concepts - real DAS data will show different characteristics

This framework provides a solid foundation for DAS ambient noise analysis using cross-correlation methods.